# CA Experiment 7 — Long-Horizon Episodic Memory for the ADA Daily-QA Agent

**Runtime:** Colab Pro **A100** + local MiniLM embeddings (free) + bounded Anthropic API (Sonnet 4.6 anchors, Sonnet 4.5 judge). Reuses the Experiment 6 checkpoints (`sft_ada_final.pt`) — an **inference-time memory-architecture** experiment, not a training run.

The Exp 6 agent is **stateless per-turn**: the 1338-token SCI overflows the 1024 window, so history is always dropped. Exp 7 gives it an external memory layer and measures long-horizon recall.

**Pipeline**
```
gen long scripts (planted anchors + lag-tagged recall probes)
  -> compact SCI (~450 tok) frees window space for a memory block
  -> run C0 (no memory) / C1 (sliding-window) / C2 (episodic-RAG)
  -> judge reliability -> analyse (recall-vs-lag, PersonaScore, cost)
```
See `CA_Experiment7_Plan.md`.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
from pathlib import Path

PROJECT_DIR = '/content/drive/MyDrive/CA_Experiment_7'
EXP6_DIR    = '/content/drive/MyDrive/CA_Experiment_6'
CKPT        = f'{EXP6_DIR}/checkpoints/sft_ada_final.pt'   # frozen Exp 6 agent
TOKENIZER   = f'{EXP6_DIR}/tokenizer/ada_bpe.json'
assert os.path.exists(PROJECT_DIR), f'Upload CA_Experiment_7 to Drive: {PROJECT_DIR}'
assert os.path.exists(CKPT), f'Exp 6 checkpoint not found: {CKPT}'

os.chdir(PROJECT_DIR)
for d in (PROJECT_DIR, EXP6_DIR):
    if d not in sys.path:
        sys.path.insert(0, d)
# Export paths so BOTH $VAR (shell, in ! cells) and {VAR} (python) resolve.
for _k in ('PROJECT_DIR', 'EXP6_DIR', 'CKPT', 'TOKENIZER'):
    os.environ[_k] = eval(_k)


def _parse_env(path):
    """Tolerant .env: BOM, blanks, comments, `export `, quotes."""
    for raw in open(path, encoding='utf-8-sig'):
        line = raw.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        if line.startswith('export '):
            line = line[len('export '):]
        k, v = line.split('=', 1)
        os.environ.setdefault(k.strip(), v.strip().strip('\'"'))


for _envp in (Path(PROJECT_DIR) / '.env', Path(EXP6_DIR) / '.env'):
    if _envp.exists():
        _parse_env(_envp)
# The anthropic SDK auto-reads ANTHROPIC_API_KEY; alias the CA key so bare
# anthropic.Anthropic() works everywhere (the scripts resolve it explicitly).
_cakey = os.environ.get('CHA_EXPERIMENT_SONNET_KEY')
if _cakey and not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = _cakey
print('cwd:', os.getcwd(), '| checkpoint ok')

In [ ]:
!pip install -q sentence-transformers anthropic python-dotenv 'tokenizers<=0.23.0' vaderSentiment qiskit qiskit-aer matplotlib

## GPU check

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime -> Change runtime type -> A100 (L4/T4 also work for inference).')
p = torch.cuda.get_device_properties(0)
print('GPU  :', p.name, f'| VRAM {p.total_memory/1e9:.0f} GB | bf16 {torch.cuda.is_bf16_supported()}')
print('Exp 7 is inference-only (frozen 321M agent + MiniLM) - any CUDA GPU works;')
print('A100/L4 keep the long C0/C1/C2 runs fast (all backbone turns generate under each condition).')

In [ ]:
# Sanity: compact SCI must fit the 1024 window with room for a memory block
from compact_sci import build_compact_system_prompt
from tokenizer_util import ADATokenizer

tok = ADATokenizer.load(TOKENIZER)
cs = build_compact_system_prompt()
n = len(tok.encode(cs))
print(f'compact SCI: {n} tokens  (full SCI is 1338; target ~450)')
assert n < 700, 'compact SCI too large — no room for a memory block'
print(cs[:500], '...')

In [ ]:
GEN_MODEL   = 'claude-sonnet-4-6'   # long-script anchors
JUDGE_MODEL = 'claude-sonnet-4-5'   # recall + PersonaScore judge (matches Exp 1-6)
os.environ['GEN_MODEL'] = GEN_MODEL

_key = os.environ.get('CHA_EXPERIMENT_SONNET_KEY') or os.environ.get('ANTHROPIC_API_KEY')
if _key:
    os.environ['ANTHROPIC_API_KEY'] = _key   # so bare anthropic.Anthropic() works everywhere too
    import anthropic
    r = anthropic.Anthropic(api_key=_key).messages.create(model=JUDGE_MODEL, max_tokens=10,
        messages=[{'role': 'user', 'content': 'Say "ok".'}])
    print('API ok:', r.content[0].text.strip(), f'(gen={GEN_MODEL} judge={JUDGE_MODEL})')
else:
    print('No API key - set CHA_EXPERIMENT_SONNET_KEY (or ANTHROPIC_API_KEY) in .env')

## Step 1 - generate long scripts
Fillers come free from the Exp 6 `persona_scripts`; Sonnet spends only on the distinctive anchors.

In [ ]:
!python gen_long_scripts.py --turn-lengths 100 300 500 --scripts-per-length 3 \
  --n-anchors 24 --lags 10 40 100 300 --model $GEN_MODEL

## Step 2 - pilot smoke test (no API, no MiniLM download)
Confirms the full generate + memory + prompt path runs on GPU before spending. Uses the offline hashing embedder and stub judge.

In [ ]:
!python evaluate_longhorizon.py --checkpoint $CKPT --condition C2 --limit 1 --max-turns 60 \
  --persona-interval 20 --stub-embedder --dry-run-judge --out-dir results_pilot

## Step 3 - full runs (C0 / C1 / C2)
Resumable. The compute-heavy step - every backbone turn generates under each condition.

In [ ]:
for cond in ('C0', 'C1', 'C2'):
    print('=' * 40, cond)
    !python evaluate_longhorizon.py --checkpoint $CKPT --condition {cond} \
      --memory-budget 300 --top-k 5 --window-n 8 --persona-interval 20 --resume

## Step 4 - judge reliability + analysis
Re-score 5% of persona probes (kappa_w >= 0.70 gate), then recall-vs-lag / persona / cost analysis + decision table.

In [ ]:
# judge reliability on a 5% sample of persona probes (Exp 1-6 gate)
import json, random, glob
from ca_assets import cohens_kappa
from evaluate import persona_judge

rows = []
for p in glob.glob('results/C2/scores_*.jsonl'):
    rows += [json.loads(l) for l in open(p) if l.strip()]
persona = [r for r in rows if r['kind'] == 'persona']
if persona:
    random.seed(1337)
    sample = random.sample(persona, max(1, int(len(persona) * 0.05)))
    a = [r['score'] for r in sample]
    b = [persona_judge(r['probe'], r['response'], r['dimension'])[0] for r in sample]
    kw = cohens_kappa(a, b, weighted=True)
    print('kappa_w', round(kw, 3), '| n', len(sample), '| gate_pass', kw >= 0.70)

In [ ]:
!python analyse_results.py --results-dir results --oow-lag 40

from IPython.display import Image, display
for f in ('exp7_recall_vs_lag', 'exp7_persona_turn_series', 'exp7_cost'):
    p = f'results/{f}.png'
    if os.path.exists(p):
        print('\n===', f, '==='); display(Image(filename=p))